In [ ]:
import basedosdados as bd
import pandas as pd



In [6]:
PROJECT_ID = "projeto-ods-12"

In [27]:
query_uf = """
SELECT 
    ano, mes, sigla_uf, tipo_consumo, consumo
FROM `basedosdados.br_mme_consumo_energia_eletrica.uf`
WHERE ano >= 2015
"""

try:
    print("Baixando dados por UF...")
    df = bd.read_sql(query_uf, billing_project_id=PROJECT_ID)

    # 2. Criamos o dicionário de mapeamento (O nosso "JOIN" manual)
    mapa_regioes = {
        'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
        'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
        'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste',
        'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
        'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
    }

    # 3. Criamos a coluna de região e agrupamos
    df['regiao'] = df['sigla_uf'].map(mapa_regioes)
    
    # Agrupamos por Região, Ano, Mês e Tipo de Consumo
    df_regioes = df.groupby(['ano', 'mes', 'regiao', 'tipo_consumo'])['consumo'].sum().reset_index()

    # 4. Criamos a coluna de data
    df_regioes['data'] = pd.to_datetime(df_regioes['ano'].astype(str) + '-' + df_regioes['mes'].astype(str) + '-01')

    # Salvando o dataset final
    df_regioes.to_csv("consumo_energia_regioes_limpo.csv", index=False)
    
    print(df_regioes.head())

except Exception as e:
    print(f"Erro: {e}")

Baixando dados por UF...


Downloading: 100%|██████████| 17496/17496 [00:01<00:00, 11672.07rows/s]

    ano  mes        regiao tipo_consumo  consumo       data
0  2015    1  Centro-Oeste       Cativo  2414412 2015-01-01
1  2015    1  Centro-Oeste    Comercial   607356 2015-01-01
2  2015    1  Centro-Oeste   Industrial   677273 2015-01-01
3  2015    1  Centro-Oeste       Outros   583718 2015-01-01
4  2015    1  Centro-Oeste  Residencial   989503 2015-01-01


In [28]:
df_regioes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3240 entries, 0 to 3239
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   ano           3240 non-null   Int64         
 1   mes           3240 non-null   Int64         
 2   regiao        3240 non-null   object        
 3   tipo_consumo  3240 non-null   object        
 4   consumo       3240 non-null   Int64         
 5   data          3240 non-null   datetime64[ns]
dtypes: Int64(3), datetime64[ns](1), object(2)
memory usage: 161.5+ KB


In [ ]:
# Os 9 estados do Nordeste
estados_ne = ('AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE')

query_ne = f"""
SELECT 
    ano, mes, sigla_uf, tipo_consumo, consumo
FROM `basedosdados.br_mme_consumo_energia_eletrica.uf`
WHERE sigla_uf IN {estados_ne}
  AND ano >= 2015
ORDER BY ano, mes, sigla_uf
"""

try:
    print("Baixando dados dos 9 estados do Nordeste...")
    df_ne = bd.read_sql(query_ne, billing_project_id=PROJECT_ID)
    
    # Criando a data para o gráfico
    df_ne['data'] = pd.to_datetime(df_ne['ano'].astype(str) + '-' + df_ne['mes'].astype(str) + '-01')
    
    # Salvando para o relatório
    df_ne.to_csv("consumo_energia_nordeste_detalhado.csv", index=False)
    print("✅ Sucesso! Agora você tem o Nordeste 'desmontado' por estado.")

except Exception as e:
    print(f"❌ Erro: {e}")

📥 Baixando dados dos 9 estados do Nordeste...


Downloading: 100%|██████████| 5832/5832 [00:00<00:00, 10596.53rows/s]

✅ Sucesso! Agora você tem o Nordeste 'desmontado' por estado.
